# Micrograd: Building a Multi-Layer Perceptron from Scratch

In this notebook we go from scalar derivatives all the way to a working multi-layer perceptron (MLP) trained by gradient descent. Every operation is built on a tiny autograd engine so you can see exactly how backpropagation works under the hood.

## Learning objectives
- Understand derivatives and numerical gradients as a sanity check.
- Implement a scalar `Value` class with automatic differentiation.
- Visualize computation graphs and verify gradients against PyTorch.
- Build `Neuron`, `Layer`, and `MLP` classes from scratch.
- Train a tiny MLP on a toy binary classification task.

## Why this matters
Modern frameworks like PyTorch and JAX hide the chain rule behind `.backward()`, but fluency in neural networks comes from seeing the machinery once. By building autograd and a small MLP by hand, you learn why gradients flow the way they do and why careful initialization, activation choices, and loss functions matter.

## Prerequisites
- Basic Python and NumPy/Matplotlib.
- Familiarity with derivatives and the chain rule.
- The `nnzero` package installed in this repo (`uv pip install -e .` or `pip install -e .`).

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from nnzero import Value, draw_dot, Neuron, Layer, MLP
from nnzero.utils import set_seed
%matplotlib inline

## 1. What is a derivative?

A derivative measures how much a function's output changes when you nudge one of its inputs. We will first check derivatives numerically before computing them automatically.

In [ ]:
def f(x):
  return 3*x**2 - 4*x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

### Numerical gradient as a sanity check

We can approximate the derivative with a tiny perturbation `h`. This is slow and approximate, but it is the gold-standard way to check that an analytical gradient implementation is correct.

In [ ]:
h = 0.000001
x = 2/3
(f(x + h) - f(x))/h

In [ ]:
# les get more complex
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
print(d)

In [ ]:
h = 0.0001

# inputs
a = 2.0
b = -3.0
c = 10.0

d1 = a*b + c
c += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2 - d1)/h)


## 2. Building a scalar autograd engine

The `Value` class wraps a scalar and records how it was produced. Each operation stores a local `_backward` function so that, starting from the final output, we can propagate gradients back to every leaf. This is reverse-mode automatic differentiation in its simplest form.

In [ ]:
class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
      
    return out
  
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out
  
  def __rmul__(self, other): # other * self
    return self * other

  def __truediv__(self, other): # self / other
    return self * other**-1

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  def __radd__(self, other): # other + self
    return self + other

  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')
    
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    
    return out
  
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')
    
    def _backward():
      self.grad += out.data * out.grad # NOTE: in the video I incorrectly used = instead of +=. Fixed here.
    out._backward = _backward
    
    return out
  
  
  def backward(self):
    
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()



### Visualizing the computation graph

We need two helpers: `trace` collects every node and edge reachable from an output, and `draw_dot` renders them with Graphviz. You already wrote these by hand in earlier lectures; now we import the maintained versions from `nnzero` so we can focus on the new concepts.

In [ ]:
from nnzero import draw_dot, trace

# draw_dot and trace are now available; they behave exactly like the hand-written versions.

### Manual backprop through a neuron

Here is the classic single-neuron example. We label every intermediate value, call `.backward()` once on the output, and let the graph fill in every `grad`. Notice that `o.grad` is initialized to `1.0` because the output is the final loss with respect to itself.

In [ ]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'
o.backward()

In [ ]:
draw_dot(o)

### Breaking `tanh` into primitive operations

`tanh` is just a composition of `exp`, `+`, `-`, `*`, and `/`. Decomposing it explicitly is a great way to confirm that our autograd engine handles multi-node graphs correctly. Every primitive operation contributes its local derivative; the chain rule chains them together automatically.

In [ ]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
# ----
e = (2*n).exp()
o = (e - 1) / (e + 1)
# ----
o.label = 'o'
o.backward()
draw_dot(o)

## 3. Checking against PyTorch

PyTorch implements the same operations in C++ and dispatches on tensors. Using double precision (`torch.double`) and scalar tensors lets us compare our `Value` gradients digit-for-digit.

In [ ]:
import torch

In [ ]:

x1 = torch.Tensor([2.0]).double()                ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()                ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()               ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()                ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()  ; b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

### 🏋️ Try it yourself #1: Add a `relu` activation to `Value`

Add a `relu()` method to the `Value` class above (or create a new standalone `Value` with the method) and use it in a small computation graph. Verify your gradients numerically with a small `h`.

*Hint:* ReLU is `max(0, x)`. Its derivative is `1` for positive inputs and `0` otherwise.

In [ ]:
# Your code here:



#### Solution

In [ ]:
# One possible implementation: attach a relu method to the local Value class.
def relu(self):
    out = Value(max(0.0, self.data), (self,), 'relu')
    def _backward():
        self.grad += (self.data > 0.0) * out.grad
    out._backward = _backward
    return out

Value.relu = relu

# Sanity check with numerical gradient
x = Value(2.0, label='x')
y = x.relu()
y.backward()
analytical = x.grad

h = 1e-6
x2 = Value(2.0 + h)
numerical = (x2.relu().data - Value(2.0).relu().data) / h
print('analytical:', analytical)
print('numerical: ', numerical)
print('match:', math.isclose(analytical, numerical, rel_tol=1e-4))


## 4. Building an MLP from scratch

Now we compose `Value` objects into `Neuron`, `Layer`, and `MLP` classes. This is the same architecture PyTorch's `nn.Linear` + `nn.Tanh` would give you, but every parameter is a `Value` so we can inspect and update it directly.

> `nnzero` already provides `Neuron`, `Layer`, and `MLP`, but we rebuild them here because writing them once is the best way to see that a neural network is just a structured graph of `Value`s.

In [ ]:

class Neuron:
  
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))
  
  def __call__(self, x):
    # w * x + b
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out
  
  def parameters(self):
    return self.w + [self.b]

class Layer:
  
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
  
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
  
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x
  
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]


In [ ]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

### Toy dataset

Four three-dimensional inputs with binary targets. The MLP will learn to separate them by minimizing a mean-squared-error loss.

In [ ]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

### Training loop

For each epoch we:
1. Run the forward pass to get predictions `ypred`.
2. Compute the mean-squared-error loss.
3. Zero out gradients from the previous step.
4. Call `.backward()` to populate every parameter's `.grad` via topological sort.
5. Update parameters with gradient descent.

> **Why topological sort?** Backprop requires that a node's gradient is fully accumulated before it is used to update its parents. A reverse topological order guarantees children are processed before their parents.

In [ ]:

for k in range(20):
  
  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  
  # backward pass
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  
  # update
  for p in n.parameters():
    p.data += -0.1 * p.grad
  
  print(k, loss.data)
  

### 🏋️ Try it yourself #2: Experiment with the optimizer

Try changing the learning rate, the network architecture (e.g. `[4, 4, 1]` vs `[8, 8, 1]`), or the loss function. What happens if you forget to zero the gradients before calling `backward()`? Add a cell below and observe.

In [ ]:
# Your experiments here:



#### Solution

In [ ]:
# Example: larger network, more iterations, smaller learning rate
set_seed(42)
n2 = MLP(3, [8, 8, 1])

for k in range(40):
    ypred = [n2(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    for p in n2.parameters():
        p.grad = 0.0
    loss.backward()

    for p in n2.parameters():
        p.data += -0.05 * p.grad

    if k % 10 == 0:
        print(k, loss.data)

print("predictions:", [v.data for v in ypred])

### Final predictions

After just a few gradient steps the network's outputs should be close to the targets `ys`.

In [ ]:
ypred

## Note on data paths

This notebook uses synthetic data. If you later load real text data (for example in the `makemore` notebooks), use paths relative to the repository root, such as `data/names.txt`. The `nnzero.utils.load_names` helper already defaults to `data/names.txt`.

## Summary

You have now built a scalar autograd engine, visualized its computation graph, verified gradients against PyTorch, and trained a tiny MLP from scratch. In the next notebooks we will scale these ideas up to tensors, embeddings, and the `makemore` character-level language model.